<a href="https://colab.research.google.com/github/LaurenMitchell-tech/uvvisml/blob/main/notebooks/test_song_models_on_Deep4Chem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch
from lightning import pytorch as pl
from pathlib import Path

from chemprop import data, featurizers, models

from google.colab import files

Cloning into 'chemprop'...
remote: Enumerating objects: 25691, done.
remote: Counting objects: 100% (448/448), done.
remote: Compressing objects: 100% (324/324), done.
remote: Total 25691 (delta 319), reused 124 (delta 123), pack-reused 25243 (from 3)
Receiving objects: 100% (25691/25691), 876.74 MiB | 28.72 MiB/s, done.
Resolving deltas: 100% (18408/18408), done.
Updating files: 100% (337/337), done.
/content/chemprop
Processing /content/chemprop
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
os.chdir('/content')
!git clone https://github.com/LaurenMitchell-tech/uvvisml
os.chdir('/content/uvvisml/uvvisml')

Cloning into 'uvvisml'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 269 (delta 89), reused 64 (delta 34), pack-reused 123 (from 1)
Receiving objects: 100% (269/269), 16.60 MiB | 8.92 MiB/s, done.
Resolving deltas: 100% (134/134), done.


## Change Test Set Input and Load

In [ ]:
test_path = 'data/splits/lambda_max_abs/deep4chem/scaffold/smiles_target_test.csv' #use data grouped the same way as the model was trained.
df_test = pd.read_csv(test_path)
df_test

,smiles,solvent,peakwavs_max
0,CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N,ClCCl,487.0
1,CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N,CS(C)=O,487.0
2,CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N,CCOC(C)=O,469.0
3,CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N,CCCCCC,432.0
4,CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N,C1CCOC1,476.0
...,...,...,...
1654,CN(C)c1ccc2c(-c3ccc(N)cc3C(=O)[O-])c3ccc(=[N+]...,O,544.0
1655,CN(C)c1ccc2c(-c3cc(N)ccc3C(=O)[O-])c3ccc(=[N+]...,O,544.0
1656,CCCCn1c2ccc(-c3ccc4c(c3)c3c5c(ccc6ccc7ccccc7c6...,ClCCl,336.0
1657,Cc1ccc(C(=O)/N=c2\oc3cccc4c5c(-c6ccccc6)oc6ccc...,ClC(Cl)Cl,385.0


## Functions

In [ ]:
def make_prediction(model_list, test_data_loader):
    all_preds_list = []

    for model in model_list:
        with torch.inference_mode():
            trainer = pl.Trainer(
                logger=None,
                enable_progress_bar=True,
                accelerator="cpu",
                devices=1
            )
            test_preds = trainer.predict(model, test_data_loader)

        test_preds = np.concatenate(test_preds, axis=0).ravel()
        all_preds_list.append(test_preds)
    return all_preds_list

In [ ]:
def plot_and_save(true_values=str, preds=str, model=str, value_type=str, save_path=str):

  '''
  true_values: name of column containing experimental x or y values
  preds: name of column containing predicted x or y values
  model: name of model used to make prediction
  save_path: path where figure will be saved

  plots a parity plot and saves the figure to the specified path
  '''

  true_ys = df_test[true_values].tolist()
  preds_y = df_test[preds].tolist()

  fig = plt.figure(figsize=(8,5))
  plt.scatter(true_ys, preds_y, color='red', marker='o', label='Parity Plot')
  ax = plt.gca()
  lims = ax.get_xlim()
  ax.plot(lims, lims, color='black')
  plt.legend()
  plt.xlabel('Experimental')
  plt.ylabel('Predicted')
  title = 'Parity Plot of ' + model + ' ' + value_type + ' Values'
  plt.title(title)
  plt.grid(True)

  with open(save_path, "wb") as f:
      pickle.dump(fig, f)

  plt.show()

  print('Plot saved as: ' + save_path)

In [ ]:
uploaded = files.upload()

Saving combined_model_0.ckpt to combined_model_0.ckpt
Saving combined_model_1.ckpt to combined_model_1.ckpt
Saving combined_model_2.ckpt to combined_model_2.ckpt


## Make Predictions

##Predict experimental peak with model trained on new combined training set

In [ ]:
#Change model input and load model
checkpoint_paths = ['/content/uvvisml/uvvisml/combined_model_2.ckpt']
models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu") #weights_only=False)
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)


all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test

[['CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N' 'ClCCl']
 ['CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N' 'CS(C)=O']
 ['CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N' 'CCOC(C)=O']
 ['CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N' 'CCCCCC']
 ['CCCCCCCCCCCCOc1cc2cc(N(C)C)ccc2cc1C=C(C#N)C#N' 'C1CCOC1']]


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/saving.py:365: Skipping 'metrics' parameter because it is not possible to safely dump to YAML.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


RuntimeError: mat1 and mat2 shapes cannot be multiplied (2392x147 and 86x300)

In [ ]:
for name, param in model.named_parameters():
    if "weight" in name:
        print(name, param.shape)
        break

message_passing.blocks.0.W_i.weight torch.Size([300, 86])


## Predict experimental peak with model trained on FluoDB training set

In [ ]:
#Change model input and load model
checkpoint_paths = ['path to model trained on FluoDB']

models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)

all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test

## Predict experimental peak with model trained on Dataset Consolidation

In [ ]:
#Change model input and load model
checkpoint_paths = ['path to model trained on Dataset_Consolidation']

models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)

all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test

In [ ]:
#for example:
plot_and_save('X', 'Pred_X', 'CIE_model_nm', 'X', 'X_Parity_CIE_model_nm_mixed_data.pkl')
files.download('X_Parity_CIE_model_nm_mixed_data.pkl')

New Data Analysis

In [ ]:
%cd data

/content/uvvisml/uvvisml/data


In [ ]:

# coding: utf-8

# # Imports

# In[1]:


import pandas as pd
import os
import numpy as np
from rdkit.Chem import AllChem as Chem
import sys
sys.path.append('..')


# In[2]:


WORK_DIR = os.getcwd()

DATA_DIR = os.path.join(WORK_DIR,'original')

data_processed_dir = os.path.join(WORK_DIR,'processed')
if not os.path.exists(data_processed_dir):
    os.makedirs(data_processed_dir)


# # Read Raw Data

# Read the raw data files, rename columns, drop NaN and extraneous information

# ### FluoDB

data_location = os.path.join(DATA_DIR, 'song/00_FluoDB_cleaned.csv')
fluodb_df = pd.read_csv(data_location)

fluodb_df = fluodb_df.rename(columns={
    'SMILES': 'smiles',
    'Solvent': 'solvent',
    'abs': 'peakwavs_max'
})

fluodb_df = fluodb_df[['smiles','solvent','peakwavs_max']].copy()
fluodb_df.dropna(subset=['smiles','solvent','peakwavs_max'], inplace=True)
fluodb_df['source'] = 'fluodb'
fluodb_df


# ### Consolidation

data_location = os.path.join(DATA_DIR, 'song/Dataset_Consolidation_cleaned.csv')
consolidation_df = pd.read_csv(data_location)

consolidation_df = consolidation_df.rename(columns={
    'SMILES': 'smiles',
    'Solvent': 'solvent',
    'Ex (nm)': 'peakwavs_max'
})

consolidation_df = consolidation_df[['smiles','solvent','peakwavs_max']].copy()
consolidation_df.dropna(subset=['smiles','solvent','peakwavs_max'], inplace=True)
consolidation_df['source'] = 'consolidation'
consolidation_df


# ### ChemFluor

# In[3]:


data_location = os.path.join(DATA_DIR, 'chem_fluor/Alldata_SMILES.xlsx')
chemfluor_df = pd.read_excel(data_location)
chemfluor_df.rename(columns={'SMILES':'smiles',"Absorption/nm":'peakwavs_max'}, inplace=True)
chemfluor_df = chemfluor_df[['smiles','solvent','peakwavs_max']].copy()
chemfluor_df.dropna(inplace=True)
chemfluor_df['source'] = 'chemfluor'
chemfluor_df


# ### DSSCDB

# In[4]:


data_location = os.path.join(DATA_DIR, 'dsscdb')
xlsx_files = [x for x in os.listdir(data_location) if x.endswith('.xlsx')]
dsscdb_df = pd.DataFrame()
for file in xlsx_files:
    file_location = os.path.join(data_location, file)
    dsscdb_df = pd.concat([dsscdb_df, pd.read_excel(file_location)], ignore_index=True, sort=True)

dsscdb_df.rename(columns={'SMILES':'smiles','SOLVENT':'solvent','ABSORPTION_MAXIMA':'peakwavs_max'}, inplace=True)
dsscdb_df = dsscdb_df[['smiles','solvent','peakwavs_max']].copy()
dsscdb_df.dropna(inplace=True)
dsscdb_df['source'] = 'dsscdb'
dsscdb_df


# ### DyeAgg

# In[5]:


data_location = os.path.join(DATA_DIR, 'dye_agg/new_dssc_Search_results.csv')
dyeagg_df = pd.read_csv(data_location, sep=';')
dyeagg_df.rename(columns={'STRUCTURE':'smiles','SOLVENT':'solvent','PEAK_ABSORPTION_SOLUTION':'peakwavs_max'},
                 inplace=True)
dyeagg_df = dyeagg_df[['smiles','solvent','peakwavs_max']].copy()
dyeagg_df.dropna(inplace=True)
dyeagg_df['source'] = 'dyeagg'
dyeagg_df


# ### CDEx

# In[6]:


data_location = os.path.join(DATA_DIR, 'jcole/paper_allDB.csv')
jcole_df = pd.read_csv(data_location)
jcole_df.rename(columns={'SMI':'smiles', "lambda_max (Exp,  nm)": 'peakwavs_max'}, inplace=True)
jcole_df = jcole_df[['smiles','solvent','peakwavs_max']].copy()
jcole_df.dropna(inplace=True)
jcole_df['source'] = 'cdex'
jcole_df


# ### Deep4Chem

# In[7]:


data_location = os.path.join(DATA_DIR, 'joung/DB_for_chromophore_Sci_Data_rev02.csv')
joung_df = pd.read_csv(data_location)

joung_df.rename(columns={'Chromophore':'smiles','Solvent':'solvent', 'Absorption max (nm)':'peakwavs_max'},
                inplace=True)

# Drop measurements where solvent == 'gas'
gas_idx = joung_df[joung_df['solvent']=='gas'].index
joung_df.drop(index=gas_idx, inplace=True)

# Drop measurements taken in solid state (original authors wrote file as chromophore == solvent for these cases)
ss_idx = joung_df[joung_df['smiles']==joung_df['solvent']].index
joung_df.drop(index=ss_idx, inplace=True)

joung_df = joung_df[['smiles','solvent','peakwavs_max']].copy()
joung_df.dropna(inplace=True)
joung_df['source'] = 'deep4chem'

joung_df


# # Convert Solvent Names to SMILES

# In[8]:


no_solvent_smiles_df = pd.concat([chemfluor_df, dsscdb_df, dyeagg_df, jcole_df])
no_solvent_smiles_df.reset_index(drop=True, inplace=True)
no_solvent_smiles_df['solvent'] = no_solvent_smiles_df['solvent'].apply(lambda x: x.lower().strip())
no_solvent_smiles_df


# In[9]:


# Exclude all solvents that are ionic, mixtures, polymers, or not clear what they are

bad_solvents = ['-','n','10 − 4–10 − 6 m','quartz','silica','pbs','pvc','ch3cn-ch2cl2','acetonitrile + tert-butanol',
                'cl','poly(ethylene glycol)s','chcl','methanol, tert-butyl alcohol-acetonitrile',
                'methanol-chloroform','tbaf','chcl3-ch3oh','ch 2 cl 2 + 0.2 mmol l − 1 bu 4 nbf','ch3oh–h2o','pmma',
                'ch3oh+chcl3','t-buoh-an','ni','ch 2 cl','acetonitrile-dmso','acetonitrile and tert-butyl alcohol',
                'sds','tert-butanol–acetonitrile','chcl3–ch3oh','barium sulfate','mecn-dcm','chloroform + methanol',
                'ch3oh-chcl3','kbr','2-methyl-2-propanol-acetonitrile']
no_solvent_smiles_df = no_solvent_smiles_df.loc[~no_solvent_smiles_df['solvent'].isin(bad_solvents)]

for bad_symbol in ['%','/',':']:
    no_solvent_smiles_df = no_solvent_smiles_df.loc[~no_solvent_smiles_df['solvent'].str.contains(bad_symbol)]

no_solvent_smiles_df


# In[10]:


# Dictionary for standardizing abbreviations and different names for the same solvents
replace_solvents_master_dict = {'acn':'acetonitrile', 'c6h6':'benzene', 'c6h12':'cyclohexane',
                                'ccl4':'carbon tetrachloride', 'ch2cl2':'dichloromethane',
                                'ch 2 cl 2':'dichloromethane','ch3cn':'acetonitrile',
                                'ch 3 cn':'acetonitrile','ch 3 oh':'methanol', 'chcl3':'chloroform',
                                'chroroform':'chloroform',
                                'chx':'chlorohexidine',
                                'ctc':'carbon tetrachloride',
                                'dichloromethane.':'dichloromethane','dcb':'1,2-dichlorobenzene',
                                'dcm':'dichloromethane',
                                'dee': 'diethyl ether',
                                'dimethylsulfoxide':'dmso', 'dimethylsufoxide':'dmso', 'dimethyl sulfoxide':'dmso',
                                'dma':'dimethylacetamide', 'dmf':'dimethylformamide',
                                'ethylacetate': 'ethyl acetate', 'etoh':'ethanol',
                                'etoac':'ethyl acetate', 'h 2 o':'water', 'h2o':'water', 'hcl':'hydrochloric acid',
                                'hex':'hexane',
                                'mch': 'methylcyclohexane',
                                'mecn':'acetonitrile', 'meoh':'methanol',
                                'me-thf':'2-methyltetrahydrofuran', 'mthf':'2-methyltetrahydrofuran',
                                'nmp':'n-methyl-2-pyrrolidone','o-c6h4cl2': 'orthodichlorobenzene',
                                'pgmea':'propylene glycol methyl ether acetate',
                                'phcl':'chlorobenzene',
                                'phcn':'benzonitrile',
                                'tfa':'trifluoroacetic acid', 'tfe':'2,2,2-trifluoroethanol',
                                'thf':'tetrahydrofuran', 'toluol':'toluene'}

# Dictionary for replacing all full solvent names with solvent SMILES
solvent_smiles_dict = {'cyclohexane': 'C1CCCCC1', 'dmso': 'CS(=O)C','ethylene glycol': 'C(CO)O',
                        'ethanol': 'CCO','methanol': 'CO','dioxane': 'C1COCCO1','pyridine': 'c1ccncc1',
                        'chlorobenzene': 'Clc1ccccc1','water': 'O','glycerin': 'OCC(O)CO',
                        'dichloromethane': 'ClCCl','carbon tetrachloride': 'ClC(Cl)(Cl)Cl',
                        'toluene': 'Cc1ccccc1','chloronaphthalene': 'Clc1cccc2ccccc12','hexane': 'CCCCCC',
                        'acetonitrile': 'CC#N','chloroform': 'ClC(Cl)Cl','benzene': 'c1ccccc1',
                        'dimethylformamide': 'CN(C)C=O','tetrahydrofuran': 'C1CCOC1',
                        'triethylamine': 'CCN(CC)CC','bromobenzene': 'Brc1ccccc1',
                        '2-methylbutane': 'CCC(C)C','2-pentanone': 'CCCC(C)=O',
                        'di-n-butyl ether': 'CCCCOCCCC','glycerol': 'OCC(O)CO','methyl formate': 'COC=O',
                        '2-propanol': 'CC(C)O','diisopropyl ether': 'CC(C)OC(C)C','1-octanol': 'CCCCCCCCO',
                        '2,2,2-trifluoroethanol': 'OCC(F)(F)F','2-methyltetrahydrofuran': 'CC1CCCO1',
                        'methylcyclohexane': 'CC1CCCCC1','1-butanol': 'CCCCO','heptane': 'CCCCCCC',
                        'ethyl acetate': 'CCOC(C)=O','1,2-dichlorobenzene': 'Clc1ccccc1Cl',
                        '1-decanol': 'CCCCCCCCCCO','formamide': 'NC=O','acetone': 'CC(C)=O',
                        'dimethylacetamide': 'CN(C)C(C)=O','o-dimethoxybenzene': 'COc1ccccc1OC',
                        'n-methylformamide': 'CNC=O','diethyl ether': 'CCOCC','1-hexanol': 'CCCCCCO',
                        '1-methyl-2-pyrrolidinone': 'CN1CCCC1=O','butyl acetate': 'CCCCOC(C)=O',
                        'tert-pentanol': 'CCC(C)(C)O','2-methyl-2-propanol': 'CC(C)(C)O',
                        '1,2-propanediol': 'CC(O)CO','1-propanol': 'CCCO','methyl acetate': 'COC(C)=O',
                        'trifluoroacetic acid': 'OC(=O)C(F)(F)F',
                        'n-methyl-2-pyrrolidone': 'CN1CCCC1=O', '1,4-dioxane': 'C1COCCO1',
                        'hydrochloric acid': 'Cl', 'benzonitrile': 'N#Cc1ccccc1',
                        'propanol': 'CCCO', 'propylene carbonate': 'CC1COC(=O)O1',
                        'benzyl alcohol': 'OCc1ccccc1', 'butanol': 'CCCCO',
                        'propylene glycol methyl ether acetate': 'COCC(C)OC(C)=O', 'isopropanol': 'CC(C)O',
                        'methylene chloride': 'ClCCl', 'n-hexane': 'CCCCCC',
                        'octene': 'CCCCCCC=C', 'mops': 'O[S](=O)(=O)CCCN1CCOCC1',
                        'tetrachloromethane': 'ClC(Cl)(Cl)Cl', 'pentane': 'CCCCC',
                        'chlorobenzene': 'Clc1ccccc1', 'acetic acid': 'CC(O)=O', '1,2-dichloroethane': 'ClCCCl',
                       'chlorohexidine':'Clc1ccc(NC(=N)NC(=N)NCCCCCCNC(=N)NC(=N)Nc2ccc(Cl)cc2)cc1',
                       'orthodichlorobenzene':'C1=CC=C(C(=C1)Cl)Cl', 'tert-butyl alcohol':'CC(C)(C)O',
               }

solvents_set = set(no_solvent_smiles_df['solvent'])
#print(len(solvents_set))

solvents_for_df_dict = {}
for solvent in solvents_set:
    if solvent in replace_solvents_master_dict.keys():
        solvent_ = replace_solvents_master_dict[solvent]
    else:
        solvent_ = solvent
    solvents_for_df_dict[solvent] = solvent_smiles_dict[solvent_]


# In[11]:


# Replace names with SMILES
no_solvent_smiles_df['solvent'] = no_solvent_smiles_df['solvent'].map(solvents_for_df_dict)
no_solvent_smiles_df


# # Combine All Sources

# In[12]:


df = pd.concat([no_solvent_smiles_df, joung_df, fluodb_df, consolidation_df])
df = df.loc[df['peakwavs_max']!='-'].copy()
df['peakwavs_max'] = df['peakwavs_max'].apply(lambda x: float(x))
df


# # Filtering

# ### Remove Molecules that Cannot be Sanitized by RDKit

# In[13]:


def sanitize_smiles(smiles):
    try:
        smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
    except:
        smiles = np.nan
    return smiles

df['smiles'] = df['smiles'].apply(lambda x: sanitize_smiles(x))
df['solvent'] = df['solvent'].apply(lambda x: sanitize_smiles(x))
df.dropna(inplace=True)
df


# ### Remove Clusters (SMILES containing ".")

# In[14]:


cluster_idx = df[df['smiles'].str.contains(r'\.')].index
#print('Removing {} rows'.format(len(cluster_idx)))
df.drop(index=cluster_idx, inplace=True)
df


# # Export DataFrame to CSV File

# In[15]:


df[['smiles','solvent','peakwavs_max','source']].to_csv(f'{data_processed_dir}/all_lambda_max_abs_including_duplicates_and_song.csv',
                                                        index=False)


# # Stats

# ### Table 2

# In[16]:


df['source'].value_counts()


# In[17]:


df.drop_duplicates(subset=['smiles','solvent'])


# In[18]:


df.groupby(['smiles','solvent']).count().query('source > 1')


# In[19]:


len(set(df['smiles']))


# In[20]:


len(set(df['solvent']))


# ### Table S1

# In[21]:


dict(df['solvent'].value_counts())

[20:43:08] Conflicting single bond directions around double bond at index 35.
[20:43:08]   BondStereo set to STEREONONE and single bond directions set to NONE.
[20:43:08] Conflicting single bond directions around double bond at index 38.
[20:43:08]   BondStereo set to STEREONONE and single bond directions set to NONE.
[20:43:10] Explicit valence for atom # 51 O, 4, is greater than permitted
[20:43:11] Can't kekulize mol.  Unkekulized atoms: 17 18 32 36 37 38 41 42 43 44 47 48
[20:43:12] Can't kekulize mol.  Unkekulized atoms: 13 14 36 40 41 42 64 65 66 67 70 71
[20:43:12] Can't kekulize mol.  Unkekulized atoms: 11 12 13 17 18 19 20 24 25 26 51 52
[20:43:12] Can't kekulize mol.  Unkekulized atoms: 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 30 31 32 33 40 44 45 46 61 62
[20:43:17] non-ring atom 1 marked aromatic
[20:43:17] non-ring atom 1 marked aromatic
[20:43:17] Explicit valence for atom # 55 O, 3, is greater than permitted
[20:43:17] WARNING: not removing hydrogen atom wit

{'ClCCl': np.int64(15422),
 'C1CCOC1': np.int64(8683),
 'CC#N': np.int64(7186),
 'ClC(Cl)Cl': np.int64(6420),
 'CO': np.int64(5667),
 'Cc1ccccc1': np.int64(5466),
 'CCO': np.int64(5342),
 'CS(C)=O': np.int64(4820),
 'O': np.int64(3588),
 'CN(C)C=O': np.int64(3293),
 'C1CCCCC1': np.int64(2492),
 'CCCCCC': np.int64(2214),
 'CCOC(C)=O': np.int64(2024),
 'C1COCCO1': np.int64(1682),
 'CC(C)=O': np.int64(1569),
 'CCOCC': np.int64(498),
 'c1ccccc1': np.int64(490),
 'CC(C)O': np.int64(363),
 'CCCCO': np.int64(342),
 'ClC(Cl)(Cl)Cl': np.int64(264),
 'CCCO': np.int64(211),
 'CCCCOCCCC': np.int64(166),
 'CCCCCCC': np.int64(155),
 'CC1CCCO1': np.int64(133),
 'c1ccncc1': np.int64(129),
 'CC1CCCCC1': np.int64(124),
 'CCCCCCCCO': np.int64(112),
 'OCCO': np.int64(109),
 'OCC(F)(F)F': np.int64(106),
 'Clc1ccccc1': np.int64(103),
 'OC(F)(F)CF': np.int64(88),
 'CC(C)(C)O': np.int64(73),
 'OCC(O)CO': np.int64(72),
 'CCCCCO': np.int64(71),
 'CCN(CC)CC': np.int64(70),
 'CCCCCCO': np.int64(69),
 'Clc1ccccc1C

In [ ]:
def count_duplicates(df, name):
    total = len(df)
    unique = df.drop_duplicates(subset=['smiles', 'solvent']).shape[0]
    duplicates = total - unique

    print(f"{name}:")
    print(f"  Total rows: {total}")
    print(f"  Unique (smiles, solvent): {unique}")
    print(f"  Duplicate rows: {duplicates}")
    print(f"  % duplicates: {duplicates / total:.2%}\n")

count_duplicates(fluodb_df, "FluoDB")
count_duplicates(consolidation_df, "Consolidation")

FluoDB:
  Total rows: 31338
  Unique (smiles, solvent): 31338
  Duplicate rows: 0
  % duplicates: 0.00%

Consolidation:
  Total rows: 21707
  Unique (smiles, solvent): 21699
  Duplicate rows: 8
  % duplicates: 0.04%



In [ ]:
total = len(df)
unique = df.drop_duplicates(subset=['smiles', 'solvent']).shape[0]
duplicates = total - unique

print("Combined dataset:")
print(f"  Total rows: {total}")
print(f"  Unique (smiles, solvent): {unique}")
print(f"  Duplicate rows: {duplicates}")
print(f"  % duplicates: {duplicates / total:.2%}")

Combined dataset:
  Total rows: 81065
  Unique (smiles, solvent): 44212
  Duplicate rows: 36853
  % duplicates: 45.46%


In [ ]:
# Duplicates across ALL data
dupes = df[df.duplicated(subset=['smiles', 'solvent'], keep=False)]

# Only rows coming from your new datasets
new_dupes = dupes[dupes['source'].isin(['fluodb', 'consolidation'])]

print(f"Duplicates involving new datasets: {len(new_dupes)}")

Duplicates involving new datasets: 35686


In [ ]:
new_dupes[['smiles', 'solvent', 'source']].sort_values(['smiles', 'solvent'])

,smiles,solvent,source
7234,B1C=Cc2c(ccc3ccccc23)N1,C1CCCCC1,fluodb
110,B1C=Cc2c(ccc3ccccc23)N1,C1CCCCC1,consolidation
111,B1C=Cc2cc3c(cc2N1)C=CBN3,C1CCCCC1,consolidation
25549,B1C=Cc2cc3ccccc3cc2N1,C1CCCCC1,fluodb
112,B1C=Cc2cc3ccccc3cc2N1,C1CCCCC1,consolidation
...,...,...,...
29412,c1csc(-c2nnnnc2-c2cccs2)c1,ClCCl,fluodb
1731,c1csc(-c2nnnnc2-c2cccs2)c1,ClCCl,consolidation
10318,c1csc(-n2c3ccccc3c3ccccc32)c1,ClCCl,fluodb
21822,c1csc(CN[C@@H]2CCCC[C@H]2NCc2cccs2)c1,CO,fluodb


In [ ]:
overlap = (
    df.groupby(['smiles', 'solvent'])['source']
    .nunique()
    .reset_index()
)

overlap = overlap[overlap['source'] > 1]

print(f"Pairs appearing in multiple datasets: {len(overlap)}")

Pairs appearing in multiple datasets: 24822


In [ ]:
# Unique pairs from old data
old_df = df[~df['source'].isin(['fluodb', 'consolidation'])]
old_pairs = set(zip(old_df.smiles, old_df.solvent))

# Unique pairs from new data
new_df = df[df['source'].isin(['fluodb', 'consolidation'])]
new_pairs = set(zip(new_df.smiles, new_df.solvent))

# Truly new chemistry
new_unique_pairs = new_pairs - old_pairs

print(f"New unique (smiles, solvent) pairs added: {len(new_unique_pairs)}")

New unique (smiles, solvent) pairs added: 17529
